In [1]:
import pandas as pd

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [29]:
fact_lab = pd.read_csv("Cleaned_datasets/fact_lab_healthcare_cleaned.csv")
state = pd.read_csv("Cleaned_datasets/dim_state_cleaned.csv")
dates = pd.read_csv("Cleaned_datasets/dim_dates_cleaned.csv")

In [14]:
fact_lab.head()

,fact_id,date_id,state_id,total_tests,positive_tests,positivity_rate,vaccination_coverage_pct,booster_coverage_pct,hospital_beds,doctors,phc_count,chc_count,icu_utilization_pct,bed_occupancy_pct,reporting_compliance_pct,turnaround_time_days,reporting_rate_pct
0,1,1,1,22918,4084.0,17.82,76.8,51.4,12451.0,1793,179,47,68.7,48.7,61.9,4.9,83.9
1,2,1,2,1323,64.0,4.85,91.6,51.6,465.0,65,10,0,77.1,38.7,98.3,2.2,96.2
2,3,1,3,28315,1734.5,14.68,71.1,37.2,8002.0,2955,119,74,76.9,55.2,72.6,1.0,71.0
3,4,1,4,90239,8653.0,9.59,79.8,26.1,10373.0,4215,979,373,83.3,60.4,98.1,4.1,98.4
4,5,1,5,23747,3931.0,16.56,90.0,52.3,7966.0,760,158,23,78.7,47.9,74.1,1.1,81.8


In [15]:
fact_lab.columns.tolist()

['fact_id',
 'date_id',
 'state_id',
 'total_tests',
 'positive_tests',
 'positivity_rate',
 'vaccination_coverage_pct',
 'booster_coverage_pct',
 'hospital_beds',
 'doctors',
 'phc_count',
 'chc_count',
 'icu_utilization_pct',
 'bed_occupancy_pct',
 'reporting_compliance_pct',
 'turnaround_time_days',
 'reporting_rate_pct']

# Healthcare Capacity KPIs

In [16]:
total_beds = fact_lab["hospital_beds"].sum()

avg_bed_occupancy = fact_lab["bed_occupancy_pct"].mean()

avg_icu_occupancy = fact_lab["icu_utilization_pct"].mean()

print("Total Hospital Beds:", total_beds)
print("Average Bed Occupancy:", round(avg_bed_occupancy, 2), "%")
print("Average ICU Occupancy:", round(avg_icu_occupancy, 2), "%")

Total Hospital Beds: 9284212.0
Average Bed Occupancy: 65.61 %
Average ICU Occupancy: 63.85 %


# Create KPI Cards

In [39]:
doctor_case_analysis = fact_lab.copy()

doctor_case_analysis["positive_cases_per_doctor"] = (
    doctor_case_analysis["positive_tests"] /
    doctor_case_analysis["doctors"].replace(0, np.nan)
)

doctor_case_analysis[
    [
        "state_id",
        "doctors",
        "positive_tests",
        "positive_cases_per_doctor"
    ]
].head()

,state_id,doctors,positive_tests,positive_cases_per_doctor
0,1,1793,4084.0,2.277747
1,2,65,64.0,0.984615
2,3,2955,1734.5,0.586971
3,4,4215,8653.0,2.052906
4,5,760,3931.0,5.172368


In [40]:
doctor_case_by_state = (
    doctor_case_analysis
    .groupby("state_id")[
        [
            "doctors",
            "positive_tests",
            "positive_cases_per_doctor"
        ]
    ]
    .mean()
    .reset_index()
)

In [41]:
doctor_case_by_state = doctor_case_by_state.merge(
    state[["state_id", "state_name"]],
    on="state_id",
    how="left"
)

doctor_case_by_state.head()

,state_id,doctors,positive_tests,positive_cases_per_doctor,state_name
0,1,2692.638889,4196.569444,1.852391,Andhra Pradesh
1,2,93.722222,123.583333,1.526911,Arunachal Pradesh
2,3,1944.666667,2672.375000,1.584841,Assam
3,4,6125.694444,8800.916667,1.712215,Bihar
4,5,1715.083333,2701.458333,1.934278,Chhattisgarh


In [42]:
top_case_load_states = (
    doctor_case_by_state
    .sort_values(
        "positive_cases_per_doctor",
        ascending=False
    )
    .head(10)
)

top_case_load_states[
    [
        "state_name",
        "doctors",
        "positive_tests",
        "positive_cases_per_doctor"
    ]
]

,state_name,doctors,positive_tests,positive_cases_per_doctor
16,Mizoram,66.777778,190.083333,3.113684
5,Goa,76.555556,135.083333,2.237778
8,Himachal Pradesh,383.305556,754.319444,2.224439
14,Manipur,166.666667,320.944444,2.091517
12,Madhya Pradesh,4336.000000,7102.541667,2.062202
24,Tripura,215.388889,427.388889,2.041710
20,Rajasthan,4250.666667,6985.236111,1.970421
4,Chhattisgarh,1715.083333,2701.458333,1.934278
21,Sikkim,36.361111,59.166667,1.920111
17,Nagaland,109.944444,178.305556,1.901864


In [45]:
dates.columns.tolist()

['date_id',
 'full_date',
 'year',
 'month_num',
 'month_name',
 'quarter',
 'year_month']

In [46]:
trend_data = fact_lab.merge(
    dates[["date_id", "full_date"]],
    on="date_id",
    how="left"
)

In [47]:
trend_data["full_date"] = pd.to_datetime(
    trend_data["full_date"]
)

In [48]:
monthly_trend = (
    trend_data
    .groupby(
        trend_data["full_date"].dt.to_period("M")
    )
    .agg(
        bed_occupancy=("bed_occupancy_pct", "mean"),
        positive_cases=("positive_tests", "sum")
    )
    .reset_index()
)

monthly_trend["full_date"] = (
    monthly_trend["full_date"]
    .astype(str)
)

monthly_trend.head()

,full_date,bed_occupancy,positive_cases
0,2022-01,63.337500,95076.0
1,2022-02,71.946875,119613.5
2,2022-03,65.081250,113229.5
3,2022-04,66.506250,102827.0
4,2022-05,64.125000,109521.0


In [44]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_case_load_states["state_name"],
        y=top_case_load_states["positive_cases_per_doctor"],
        name="Positive Cases per Doctor",
        marker_color="purple",
        hovertemplate=
            "<b>%{x}</b><br>" +
            "Positive Cases per Doctor: %{y:.2f}<br>" +
            "<extra></extra>"
    )
)

fig.update_layout(
    title="Top 10 States by Positive Cases per Doctor",
    xaxis_title="State",
    yaxis_title="Positive Cases per Doctor",
    xaxis_tickangle=-45,
    template="plotly_white",
    height=550,
    hovermode="x"
)

fig.show(renderer="iframe")

In [49]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=monthly_trend["full_date"],
        y=monthly_trend["bed_occupancy"],
        mode="lines+markers",
        name="Bed Occupancy",
        line=dict(
            color="royalblue",
            width=3
        ),
        marker=dict(
            size=7
        ),
        hovertemplate=
            "<b>Month:</b> %{x}<br>" +
            "<b>Bed Occupancy:</b> %{y:.2f}%<br>" +
            "<extra></extra>"
    )
)

fig.add_trace(
    go.Scatter(
        x=monthly_trend["full_date"],
        y=monthly_trend["positive_cases"],
        mode="lines+markers",
        name="Positive Cases",
        yaxis="y2",
        line=dict(
            color="crimson",
            width=3
        ),
        marker=dict(
            size=7
        ),
        hovertemplate=
            "<b>Month:</b> %{x}<br>" +
            "<b>Positive Cases:</b> %{y:,.0f}<br>" +
            "<extra></extra>"
    )
)

fig.update_layout(
    title="Monthly Bed Occupancy vs Positive Case Surges",

    xaxis=dict(
        title="Month",
        tickangle=-45
    ),

    yaxis=dict(
        title="Average Bed Occupancy (%)"
    ),

    yaxis2=dict(
        title="Positive Cases",
        overlaying="y",
        side="right"
    ),

    hovermode="x unified",

    template="plotly_white",

    height=600,

    width=1100
)

fig.show(renderer="iframe")

In [38]:
fig = go.Figure(
    go.Indicator(
        mode="gauge+number",
        value=avg_bed_occupancy,
        title={
            "text": "Average Bed Occupancy"
        },
        number={
            "suffix": "%",
            "valueformat": ".2f"
        },
        gauge={
            "axis": {
                "range": [0, 100]
            },
            "steps": [
                {
                    "range": [0, 50],
                    "color": "lightgreen"
                },
                {
                    "range": [50, 80],
                    "color": "gold"
                },
                {
                    "range": [80, 100],
                    "color": "lightcoral"
                }
            ],
            "threshold": {
                "line": {
                    "color": "black",
                    "width": 4
                },
                "value": avg_bed_occupancy
            }
        }
    )
)

fig.update_layout(
    title="Hospital Bed Occupancy Gauge",
    height=450,
    template="plotly_white"
)

fig.show(renderer="iframe")